# Targeted Synthetic Claim Scenario Generation

This notebook focuses on generating **synthetic claim scenarios** to augment our dataset. We aim to generate borderline cases and underrepresented patterns that challenge the ML model.

## Motivation
- The ML model struggles with certain edge cases (e.g., ambiguous descriptions, high excess-to-RRP ratios where the damage doesn't sound severe).
- Our data is imbalanced (~84% approved, ~16% declined).
- By prompting a Generative AI model to produce synthetic examples of **sparse decline/border strata** (quantitative cross-tabs) plus optional **persona marginals**, we can augment training and stress the RF near decision boundaries.

In [4]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

_CWD = Path.cwd().resolve()
PROJECT_ROOT = _CWD.parent if _CWD.name == "notebooks" else _CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from app.genai.llm import generate_content

df = pd.read_csv(PROJECT_ROOT / "data" / "claim_use_case_dataset_enriched.csv")
print(f"Loaded {len(df)} base claims.")

Loaded 2880 base claims.


## 1. Identify Underrepresented Patterns

### 1a. Stratified patterns (excluding persona)

We quantify **patterns** as discrete cells in a cross-tab:  
`claimType` × `coverage` × `channel` × **`fee_to_rrp` tertile** (economics) × **`coverage_duration_tier`** (policy span band).

#### Probabilities vs labels — raw data has no “probability bands”

The enriched CSV gives each claim **`approved`** (the realised outcome: 1 = approved, 0 = declined). It does **not** include a `probability_approved` column: operational data records decisions, not an internal ML score.

To study **model‑borderline** behaviour we **re‑score** every loaded row with the trained approval pipeline (`predict_batch` in `app/ml/predict.py`): claim rows are feature‑engineered the same way as at train/inference time, `approval_model.joblib` is loaded from the configured artifacts directory, and **`predict_proba`** returns \(P(\text{approve} \mid \mathbf{x})\) for the positive (“approved”) class. That vector is attached as **`probability_approved`** only in this notebook’s working dataframe (`§1a` code cell), not as part of the raw extract.

The **border band** `[BORDER_LO, BORDER_HI]` (defaults **0.40–0.60**) is likewise **not** something “known from” the CSV. It is an **analyst‑chosen** slice of the RF score where we say “the model is ambiguous enough to treat this as near the decision boundary” — useful for finding sparse strata full of hard cases and for telling the LLM what “near \(p \approx 0.5\)” means in this project. You could tighten (e.g. 0.45–0.55) or widen (e.g. 0.35–0.65); we keep fixed constants so strata rankings and prompt wording stay comparable across runs.

The border band is only meaningful if you trust the RF as a guide to ambiguity — limitations are spelled out under **Caveats** (markdown cell immediately after the §1a computation).

Per cell \(c\) we compute:

| Metric | Meaning |
|--------|---------|
| **n_decl** | Count of declines (`approved == 0`) in the cell |
| **n_border** | Count with RF **`probability_approved` ∈ \[0.40, 0.60\] ** |
| **decline_rate** | n_decl / n_total |
| **lift_decl** | decline_rate ÷ portfolio decline rate (≈ independence baseline) |
| **Pearson residual (decl)** | \((n_{\text{decl}} - E) / \sqrt{E}\) with \(E = n_{\text{total}} \times p_{\text{decl, global}}\) |

**Rare denial patterns** → sort by ascending **n_decl** (with minimum \(n_{\text{total}}\) support so ratios are meaningful).  
**Rare borderline patterns** → sort by ascending **n_border**.

#### The three strata excerpts (Tables A–C)

Each printed block is a **sorted slice** of the full grid (exported to **`data/pattern_strata_denial_borderline.csv`**). Summary rules:

| Printed block | Selection rule (after filters) | What “rare” / “hot” means here |
|---------------|-------------------------------|----------------------------------|
| **Rare decline patterns** | Cells with `n_total ≥ MIN_CELL_SUPPORT`, `n_decl ≥ 1`, sort **`n_decl` ascending** (tie-break **`n_total` ascending**), **`head(14)`** | Among qualifying cells, **few realised declines in that bucket**. Not the same as “globally unusual claim type” alone — low `n_decl` with modest `n_total` is the intended signal. |
| **Rare RF-borderline-density patterns** | Same support filter, `n_border ≥ 1`, sort **`n_border` ascending**, **`head(14)`** | **Few rows** in that cell land in the RF band `[BORDER_LO, BORDER_HI]`. Sparse **count** of ambiguous scores — not proof every narrative in the cell is qualitatively borderline. |
| **High `lift_decl` cells** | `n_decl ≥ 5`, support filter, sort **`lift_decl` descending**, **`head(8)`** | **Decline rate in the cell** is elevated vs portfolio baseline (`lift_decl ≈ 1` if unrelated). Suited to **denial-heavy** vignettes even when `n_decl` is not minimal. |

The full grid is in **`data/pattern_strata_denial_borderline.csv`** (hundreds of rows); the LLM only receives these **top‑k excerpts** embedded in the prompt.

#### Validating “borderline” and “rare”

- **Borderline (RF)** — Each row counted in **`n_border`** has **`probability_approved ∈ [BORDER_LO, BORDER_HI]`** by construction (those rows were flagged before aggregation). Whether that band is **informative** is a separate question: compare the band to the **global distribution** of RF scores (§1a prints quantiles and share in-band). Very narrow bands near 0 or 1 behave differently from a band around the score median.
- **Rare (tables)** — “Rare” is **operational**: fixed **`MIN_CELL_SUPPORT`**, sort keys, and **`head(k)`** caps. It is **not** a statistical significance test. Inspect the CSV, change **`MIN_CELL_SUPPORT`**, or widen/narrow **`head`** if excerpts feel noisy or redundant.
- **After synthesis** — Run **`predict_batch`** on synthetic rows: fraction in the RF band and decline mix vs historic slices gives a coarse check that the generator followed the intent.

#### Does the LLM see how rare things are?

**Partially.** The prompt pastes each table as **plain text**, including **`n_total`, `n_decl`, `n_border`, `mean_p_approve`, `lift_decl`**, so the model can reason about **sparsity inside those rows**. It does **not** automatically see strata that were **left out** of the **`head(k)`** slice or the **total number of grid cells** unless you add that explicitly (the §2 code cell **prints the exact prompt** so you can audit what reached the model).

After running the **§1a computation** cell (scores + CSV + `patterns_sparse_decl` / `patterns_sparse_border` / `hot`), read **Caveats**, then run **Table A, Table B, Table C** (each has interpretation markdown + a display cell).

Results: computation cell writes **`data/pattern_strata_denial_borderline.csv`**; excerpts render in the Table A/B/C code cells.

### 1b. Personas among declines (supplementary)

Optional marginal view for narrative personas — orthogonal to §1a.

In [5]:
import numpy as np

from app.ml.predict import predict_batch


def fee_to_rrp_tertiles(fee_rrp: pd.Series) -> pd.Series:
    """Tertiles by rank pct (handles ties cleanly)."""
    x = pd.to_numeric(fee_rrp, errors="coerce").rank(method="average", pct=True)
    return pd.cut(
        x,
        bins=[-0.001, 1 / 3, 2 / 3, 1.001],
        labels=["fee_ratio_low", "fee_ratio_mid", "fee_ratio_high"],
        include_lowest=True,
    )


BORDER_LO, BORDER_HI = 0.40, 0.60
MIN_CELL_SUPPORT = 10  # minimum rows per cell before we trust pattern ranks

GROUP_COLS = ["claimType", "coverage", "channel", "_fee_tert", "coverage_duration_tier"]

# Shared scoring — reused in §2 LLM prompt
_, PROBA_FULL = predict_batch(df)

_work = pd.DataFrame(
    {
        "claimType": df["claimType"],
        "coverage": df["coverage"],
        "channel": df["channel"],
        "fee_to_rrp": pd.to_numeric(df["fee_to_rrp"], errors="coerce"),
        "coverage_duration_tier": df.get("coverage_duration_tier"),
        "approved": df["approved"],
        "probability_approved": PROBA_FULL,
        "_decl": (df["approved"].to_numpy() == 0).astype(int),
        "_border": ((PROBA_FULL >= BORDER_LO) & (PROBA_FULL <= BORDER_HI)).astype(int),
    },
    index=df.index.copy(),
)
_work["_fee_tert"] = fee_to_rrp_tertiles(_work["fee_to_rrp"])

p_decl_global = float(_work["_decl"].mean())
n_total_global = len(_work)

cell = (
    _work.groupby(GROUP_COLS, dropna=False, observed=False)
    .agg(
        n_total=("approved", "size"),
        n_decl=("_decl", "sum"),
        n_border=("_border", "sum"),
        mean_p_approve=("probability_approved", "mean"),
    )
    .reset_index()
)

cell["decline_rate"] = cell["n_decl"] / cell["n_total"].clip(lower=1)
cell["lift_decl"] = np.where(p_decl_global > 0, cell["decline_rate"] / p_decl_global, np.nan)
exp_decl = cell["n_total"] * p_decl_global
cell["pearson_residual_decl"] = (cell["n_decl"] - exp_decl) / np.sqrt(np.clip(exp_decl, 1e-9, None))

_p_full = np.asarray(PROBA_FULL, dtype=float)
_q = np.quantile(_p_full, [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
print("\n--- RF P(approve) on full df (sanity vs border band) ---")
print(f"  quantiles [5%,10%,25%,50%,75%,90%,95%]: {_q.round(4)}")
print(
    f"  share in band [{BORDER_LO},{BORDER_HI}]: "
    f"{_work['_border'].mean():.4f} ({int(_work['_border'].sum())} / {len(_work)} rows)"
)
print("\n--- Three-table excerpts (matches text injected into LLM) ---")
print(
    f"  A Rare declines: n_total>={MIN_CELL_SUPPORT}, n_decl>=1, "
    "sort n_decl↑ then n_total↑, head 14"
)
print(
    f"  B Rare border-density: n_total>={MIN_CELL_SUPPORT}, n_border>=1, "
    "sort n_border↑ then n_total↑, head 14"
)
print(
    f"  C High lift_decl: n_decl>=5, n_total>={MIN_CELL_SUPPORT}, "
    "sort lift_decl↓, head 8"
)
print(f"  Strata grid size: {len(cell)} rows → CSV; LLM sees only these slices + prompt prose.")

_pat_csv = PROJECT_ROOT / "data" / "pattern_strata_denial_borderline.csv"
cell.sort_values(["n_decl", "n_border"], ascending=[True, True]).to_csv(_pat_csv, index=False)
print(f"Wrote strata table ({len(cell)} rows) → {_pat_csv.relative_to(PROJECT_ROOT)}")

# Rare observed declines among cells with enough support
msk = (cell["n_decl"] >= 1) & (cell["n_total"] >= MIN_CELL_SUPPORT)
patterns_sparse_decl = (
    cell.loc[msk].sort_values(["n_decl", "n_total"], ascending=[True, True]).head(14).reset_index(drop=True)
)

msk_b = (cell["n_border"] >= 1) & (cell["n_total"] >= MIN_CELL_SUPPORT)
patterns_sparse_border = (
    cell.loc[msk_b].sort_values(["n_border", "n_total"], ascending=[True, True]).head(14).reset_index(drop=True)
)

# Denial rate uplift vs portfolio (interesting for synthetic "hard decline" vignettes)
hot = (
    cell[
        (cell["n_decl"] >= 5)
        & (cell["n_total"] >= MIN_CELL_SUPPORT)
        & (cell["lift_decl"].notna())
    ]
    .sort_values("lift_decl", ascending=False)
    .head(8)
    .reset_index(drop=True)
)

META_BLOCK = (
    f"n={n_total_global}; portfolio P(declined)={p_decl_global:.4f}; "
    f"RF border band [{BORDER_LO:.2f}, {BORDER_HI:.2f}]: {_work['_border'].sum()} rows "
    f"({100.0 * _work['_border'].mean():.1f}%)."
)

SPARSE_DECL_BLOCK = patterns_sparse_decl.to_string(index=False)
SPARSE_BORDER_BLOCK = patterns_sparse_border.to_string(index=False)
DENIAL_HOTSPOT_BLOCK = hot.to_string(index=False)

print(META_BLOCK, "\n")
print("Strata excerpts: run the caveats cell, then each Table A/B/C pair below.")


Wrote strata table (480 rows) → data\pattern_strata_denial_borderline.csv
n=2880; portfolio P(declined)=0.1573; RF border band [0.40, 0.60]: 205 rows (7.1%). 

Rare decline patterns (sorted by ascending n_decl, min support)



c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,claimType,coverage,channel,_fee_tert,coverage_duration_tier,n_total,n_decl,n_border,mean_p_approve,decline_rate,lift_decl,pearson_residual_decl
0,Accidental Damage,ADLD/THEFT,Email,fee_ratio_mid,tier_appx_12mo,11,1,2,0.816709,0.090909,0.577965,-0.555134
1,Liquid Damage,ADLD,Online Portal,fee_ratio_low,tier_appx_12mo,11,2,1,0.797273,0.181818,1.155930,0.205106
2,Accidental Damage,ADLD/THEFT,Online Portal,fee_ratio_high,tier_appx_24mo,12,2,0,0.763333,0.166667,1.059603,0.081886
3,Accidental Damage,ADLD,Phone Call,fee_ratio_high,tier_appx_12mo,11,3,0,0.729091,0.272727,1.733895,0.965347
4,Accidental Damage,ADLD/THEFT,Online Portal,fee_ratio_low,tier_appx_24mo,23,3,1,0.892174,0.130435,0.829254,-0.324763
5,Accidental Damage,ADLD,Online Portal,fee_ratio_high,tier_appx_6mo,26,4,3,0.836154,0.153846,0.978095,-0.044298
6,Accidental Damage,ADLD,Online Portal,fee_ratio_mid,tier_appx_6mo,40,4,3,0.847500,0.100000,0.635762,-0.913626
7,Liquid Damage,ADLD,Online Portal,fee_ratio_high,tier_appx_12mo,13,5,3,0.713077,0.384615,2.445237,2.066634
8,Accidental Damage,ADLD,Online Portal,fee_ratio_low,outlier_lt120d,51,5,0,0.903922,0.098039,0.623296,-1.066936
9,Accidental Damage,ADLD,Email,fee_ratio_high,tier_appx_12mo,26,6,1,0.750689,0.230769,1.467142,0.944688


Rare RF-borderline patterns (sorted by ascending n_border, min support)



,claimType,coverage,channel,_fee_tert,coverage_duration_tier,n_total,n_decl,n_border,mean_p_approve,decline_rate,lift_decl,pearson_residual_decl
0,Liquid Damage,ADLD,Online Portal,fee_ratio_low,tier_appx_12mo,11,2,1,0.797273,0.181818,1.155930,0.205106
1,Accidental Damage,ADLD/THEFT,Online Portal,fee_ratio_low,tier_appx_24mo,23,3,1,0.892174,0.130435,0.829254,-0.324763
2,Accidental Damage,ADLD,Email,fee_ratio_high,tier_appx_12mo,26,6,1,0.750689,0.230769,1.467142,0.944688
3,Accidental Damage,ADLD/THEFT,Email,fee_ratio_mid,tier_appx_12mo,11,1,2,0.816709,0.090909,0.577965,-0.555134
4,Accidental Damage,ADLD,Email,fee_ratio_low,tier_appx_12mo,40,6,2,0.906239,0.150000,0.953642,-0.116280
5,Accidental Damage,ADLD,Online Portal,fee_ratio_low,tier_appx_6mo,84,6,2,0.899405,0.071429,0.454115,-1.984236
6,Liquid Damage,ADLD,Online Portal,fee_ratio_high,tier_appx_12mo,13,5,3,0.713077,0.384615,2.445237,2.066634
7,Accidental Damage,ADLD,Online Portal,fee_ratio_mid,outlier_lt120d,24,8,3,0.847083,0.333333,2.119205,2.174544
8,Accidental Damage,ADLD,Online Portal,fee_ratio_high,tier_appx_6mo,26,4,3,0.836154,0.153846,0.978095,-0.044298
9,Theft,ADLD/THEFT,Online Portal,fee_ratio_mid,tier_appx_12mo,28,9,3,0.730030,0.321429,2.043519,2.189941


High lift_decl cells (dense declines vs portfolio expectation)



,claimType,coverage,channel,_fee_tert,coverage_duration_tier,n_total,n_decl,n_border,mean_p_approve,decline_rate,lift_decl,pearson_residual_decl
0,Liquid Damage,ADLD,Online Portal,fee_ratio_high,tier_appx_12mo,13,5,3,0.713077,0.384615,2.445237,2.066634
1,Accidental Damage,ADLD,Online Portal,fee_ratio_mid,outlier_lt120d,24,8,3,0.847083,0.333333,2.119205,2.174544
2,Theft,ADLD/THEFT,Online Portal,fee_ratio_mid,tier_appx_12mo,28,9,3,0.730030,0.321429,2.043519,2.189941
3,Accidental Damage,ADLD,Online Portal,fee_ratio_high,tier_appx_24mo,75,19,12,0.793333,0.253333,1.610596,2.097188
4,Accidental Damage,ADLD,Email,fee_ratio_high,tier_appx_12mo,26,6,1,0.750689,0.230769,1.467142,0.944688
5,Accidental Damage,ADLD,Online Portal,fee_ratio_high,tier_appx_12mo,599,128,46,0.780649,0.213689,1.358556,3.480349
6,Accidental Damage,ADLD/THEFT,Online Portal,fee_ratio_high,tier_appx_12mo,96,20,11,0.788198,0.208333,1.324503,1.260978
7,Theft,ADLD/THEFT,Online Portal,fee_ratio_high,tier_appx_12mo,47,9,3,0.816809,0.191489,1.217416,0.591143


### Caveats: synthetic rows and using the RF as a probability baseline

**Mock / LLM-generated claims** look realistic but are **not** sampled from the real claims process. Joint distributions (coverage × economics × narrative tone), fraud or anti-selection, policy wording, channel effects, and operational QA are **not guaranteed**. Treat synthetic rows as **stress-test candidates** until you validate them (e.g. **`predict_batch`** on drafts, leakage checks, manual review).

**RF scores define “borderline” here** (`n_border`, the `[BORDER_LO, BORDER_HI]` band, and calibration snippets in §2). That is informative only if the Random Forest is **strong enough and sufficiently calibrated** for your use case. If the model is weak, overfit, stale, or miscalibrated:

- Scores may diverge from human adjudication or true risk.
- Strata ranked by **`n_border`** or “near 0.5” may highlight **artifacts**, not genuine ambiguity.
- Training heavily on RF-guided synthetic data can **amplify the model’s own blind spots**.

Prefer calibration / reliability checks and holdout metrics before trusting RF-guided synthesis as a stand-in for real-world uncertainty.


#### Table A — Rare decline patterns

**How to read:** Each row is one stratum with **`n_decl ≥ 1`**, **`n_total ≥ MIN_CELL_SUPPORT`**, sorted so **small `n_decl`** rises to the top (tie-break smaller **`n_total`**), then **`head(14)`**.

**Interpretation:** Buckets where **history shows few declines** — useful seeds for **sparse-decline** synthetic stories. That does **not** automatically mean “low risk”; it may reflect **small volume** or past leniency. Use **`decline_rate`** and **`lift_decl`** together: a row can have tiny **`n_decl`** but still sit near portfolio decline rate if **`n_total`** is small (support filter reduces but does not remove that).


In [ ]:
print("Table A — Rare decline patterns (ascending n_decl, min support)\n")
display(patterns_sparse_decl)


#### Table B — Rare RF-borderline-density patterns

**How to read:** Strata with **`n_border ≥ 1`**, same **`MIN_CELL_SUPPORT`**, sorted by **ascending `n_border`** (tie-break **`n_total`**), **`head(14)`**.

**Interpretation:** Cells where **few claims receive RF scores in the border band** — i.e. the **current model rarely outputs “ambiguous” probabilities** for that slice. Good prompts for **forcing** narratives toward ambiguity **only if** you trust the RF (see caveats). **`mean_p_approve`** averages scores over **all** rows in the cell (not only borderline rows), so it can stay high while **`n_border`** is low.


In [ ]:
print("Table B — Rare RF-borderline-density patterns (ascending n_border, min support)\n")
display(patterns_sparse_border)


#### Table C — High `lift_decl` cells

**How to read:** **`n_decl ≥ 5`** (needs some mass), **`n_total ≥ MIN_CELL_SUPPORT`**, sort **`lift_decl` descending**, **`head(8)`**.

**Interpretation:** **Denial-heavy buckets relative to portfolio baseline** — strong anchors for synthetic **declined** or hard-adjudication vignettes. Unlike Table A, we **require** several declines so rates are less brittle. **`pearson_residual_decl`** in the CSV measures excess declines vs a simple independence expectation; large positives line up with “hot” denial cells.


In [ ]:
print("Table C — High lift_decl cells (decline rate vs portfolio)\n")
display(hot)


In [6]:
# §1b supplementary — orthogonal to strata in §1a
declined = df[df["approved"] == 0]
print("Persona marginal among declined rows:")
print(declined["persona"].value_counts())

Persona marginal among declined rows:
persona
The Clumsy Dropper (Standard Accidental)    312
The Professional (Workplace Accident)        29
The Commuter (Transit Damage)                28
The Victim (Theft/Loss)                      25
The Active/Sporty (Action Damage)            23
The Family/Pet Owner (Chaos Damage)          21
The Unlucky Spiller (Liquid Damage)          15
Name: count, dtype: int64


## 2. Prompt Engineering Strategy for Synthetic Generation

We want the LLM to generate structured JSON matching our CSV schema. Our prompt strategy is:
1. **Context Initialization**: Give the LLM the role of an expert claims data simulator.
2. **Schema Definition**: Clearly outline the output format (JSON list) and the required fields (`claimType`, `channel`, `excessFee`, `rrp`, `issue_desc_en`, `persona`, `status`).
3. **Targeted Constraints**: Explicitly instruct the LLM to generate borderline cases (e.g., claims with high list prices but suspicious, ambiguous damage descriptions that should be declined).
4. **Persona focus (grounded)**: **Inject** the canonical persona list from `app.ml.persona_labels.PERSONA_CANONICAL`, the actual **declined** `persona` value counts from §1, and a data-driven “underrepresented” subset (rare among declined rows). **Do not** rely on the model to guess which labels exist.
5. **Model borderline band**: Uses `PROBA_FULL` from §1a (`predict_batch` once on `df`). Those scores are **model outputs**, not columns in the source CSV; the numeric endpoints `[BORDER_LO, BORDER_HI]` are chosen for this notebook (see §1a *Probabilities vs labels*).
6. **Strata grounding (excluding persona):** Injects the printed tables from §1a—rare-decline patterns, rare borderline-density patterns, and high-`lift_decl` cells—into the Gemini prompt (`data/pattern_strata_denial_borderline.csv` mirrors the join key). See §1a *The three on-screen tables* for how each excerpt is defined and what “rare” means.

Run cells in order: load → §1a computation → **Caveats** → **Tables A–C** → §1b (optional) → §2.

The **next code cell prints the full prompt string** (character count plus exact text) immediately before calling the LLM so you can audit counts/tables/persona blocks the model actually receives.

**Exports:** **`data/pattern_strata_denial_borderline.csv`** (§1a) and **`data/synthetic_incremental_last_run.csv`** (this section, overwrite).

In [7]:
from app.ml.persona_labels import PERSONA_CANONICAL, normalize_persona_label
from app.ml.predict import predict_batch

canonical_block = "\n".join(f"  - {p}" for p in PERSONA_CANONICAL)

_decl = declined.copy()
_decl["_pn"] = _decl["persona"].map(lambda x: normalize_persona_label(x))
decl_by_persona = _decl["_pn"].value_counts().sort_values(ascending=True)
_cut = float(decl_by_persona.median()) if len(decl_by_persona) else 0.0
underrep_personas = [p for p, c in decl_by_persona.items() if c <= _cut]
underrep_txt = ", ".join(underrep_personas) if underrep_personas else "(none)"

proba_arr = globals().get("PROBA_FULL")
_lo = globals().get("BORDER_LO", 0.40)
_hi = globals().get("BORDER_HI", 0.60)
if proba_arr is None:
    _, proba_arr = predict_batch(df)

pser = pd.Series(list(proba_arr), index=df.index, dtype=float)
in_band = (pser >= _lo) & (pser <= _hi)
borderline_n = int(in_band.sum())
borderline_pct = 100.0 * borderline_n / max(len(df), 1)

_br = df.loc[in_band]
ix_take = _br.index[:6]
mini = _br[["claimType", "coverage", "channel", "excessFee", "rrp"]].head(6).copy()
mini["probability_approved"] = pser.loc[mini.index].tolist()
border_preview = mini.to_string(index=False)

sparse_decl_txt = globals().get("SPARSE_DECL_BLOCK") or "(run §1a cell first)"
sparse_border_txt = globals().get("SPARSE_BORDER_BLOCK") or "(run §1a cell first)"
denial_lift_txt = globals().get("DENIAL_HOTSPOT_BLOCK") or "(run §1a cell first)"
meta_txt = globals().get("META_BLOCK") or ""

declined_counts_block = declined["persona"].map(normalize_persona_label).value_counts().to_string()

prompt = f"""You are an expert data simulator for a mobile device insurance provider.
Your task is to generate 5 highly realistic synthetic claim scenarios (structured + narrative).

Ground each row using:
- **Dimensional strata** (excluding persona semantics): reuse exact buckets from §1a for claimType × coverage × channel × fee-ratio tertile (labels `fee_ratio_low` / `fee_ratio_mid` / `fee_ratio_high` computed from tertiles of fee_to_rrp) × coverage_duration_tier.
- **Persona**: each object must carry one canonical string from the persona list below.

Requirements:
1. At least **2** scenarios MUST match rows from \"Rare declines\" (minimal n_decl) while remaining realistic.
2. At least **2** scenarios MUST mimic \"Rare RF-border strata\" mixes so the hypothetical claim would reasonably land near p(approve) in [{_lo:.2f},{_hi:.2f}] with ambiguous facts.
3. At least **1** scenario SHOULD echo a \"High lift_decl\" cell (elevated denies vs portfolio) with a plausible decline story.

Prefer sparse-declined personas in the narrative wording: {underrep_txt}

Canonical persona strings (persona field = exact match):
{canonical_block}

Declined-only persona counts (normalized):
{declined_counts_block}

Portfolio + RF border summary: {meta_txt}

RF calibration examples (historic rows whose p(approve) fell in-band; mimic structure, don't copy verbatim):
{border_preview}

--- §1a tables (omit persona):

Rare declines — ascending n_decl (min_support ≥10 per cell):

{sparse_decl_txt}

Rare borderline-density strata — ascending n_border:

{sparse_border_txt}

High decline-rate uplift vs portfolio:

{denial_lift_txt}

Output only a JSON array; no prose outside JSON; objects use keys claimType, channel, excessFee, rrp, issue_desc_en, persona, status.
"""

print(f"\n=== LLM request: prompt length = {len(prompt):,} characters ===")
print("--- PROMPT SENT TO LLM (exact string) ---")
print(prompt)
print("--- END PROMPT ---\n")

_export_path = PROJECT_ROOT / "data" / "synthetic_incremental_last_run.csv"
try:
    response_text = generate_content(prompt)
    if response_text.strip().startswith("```json"):
        response_text = response_text.strip()[7:]
    if response_text.strip().startswith("```"):
        response_text = response_text.strip()[3:]
    if response_text.rstrip().endswith("```"):
        response_text = response_text.rstrip()[:-3]

    synthetic_data = json.loads(response_text.strip())
    synthetic_df = pd.DataFrame(synthetic_data)
    synthetic_df.to_csv(_export_path, index=False)
    print("Successfully generated synthetic claims; wrote", _export_path)
    display(synthetic_df)
except Exception as e:
    print("Failed to generate or parse LLM output:", e)
    print("\n--- Mock LLM Output ---")
    mock_output = [
        {
            "claimType": "Theft",
            "channel": "Online Portal",
            "excessFee": 500,
            "rrp": 18000,
            "issue_desc_en": "I left my phone on the table at a busy cafe while I went to the restroom for 15 minutes. When I came back, the phone was gone. I don't have a police report yet because I thought I might have just misplaced it.",
            "persona": "The Victim (Theft/Loss)",
            "status": "Declined",
        }
    ]
    display(pd.DataFrame(mock_output))


Successfully generated synthetic claims; wrote C:\coding\boltech\data\synthetic_incremental_last_run.csv


,claimType,channel,excessFee,rrp,issue_desc_en,persona,status
0,Liquid Damage,Online Portal,150.0,1200.0,The user reported that their device was submer...,The Unlucky Spiller (Liquid Damage),Approved
1,Accidental Damage,Email,250.0,800.0,Device screen shattered after being dropped on...,The Clumsy Dropper (Standard Accidental),Approved
2,Theft,Online Portal,300.0,1100.0,Customer claims the device was stolen from the...,The Victim (Theft/Loss),Declined
3,Accidental Damage,Online Portal,450.0,900.0,The device was crushed during a mountain bikin...,The Active/Sporty (Action Damage),Approved
4,Accidental Damage,Online Portal,200.0,1400.0,The device was found damaged after being left ...,The Family/Pet Owner (Chaos Damage),Declined


## 3. Augmenting the Original Dataset

Once generated, these synthetic rows can be concatenated with the original dataset to provide a richer, more balanced training set for the next iteration of the ML model.